# 01 Video audit (all cameras)

Lists every video under `data/raw/`, reads duration / resolution / fps with ffprobe, works out the sampled-frame and chunk counts, and saves `manifests/video_manifest.csv`. Then preview any frame by elapsed second.

Expected layout: `data/raw/cam1/A01_<YYYYMMDDHHMMSS>.mp4`, `cam2/A02_...`, `cam3/A03_...`, `cam4/A04_...`

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/CMZprojectphase2")
RAW = PROJECT / "data" / "raw"
MANIFESTS = PROJECT / "manifests"
MANIFESTS.mkdir(parents=True, exist_ok=True)

videos = sorted(RAW.rglob("*.mp4"))
print(f"{len(videos)} videos found")
for v in videos:
    print(" ", v.relative_to(RAW).as_posix())

In [ ]:
import json
import math
import re
import subprocess
from datetime import datetime

import pandas as pd

SAMPLE_INTERVAL_SECONDS = 1
CHUNK_SECONDS = 300  # five minutes per chunk


def probe(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "format=duration:stream=codec_name,width,height,r_frame_rate",
         "-of", "json", str(path)],
        capture_output=True, text=True, check=True,
    ).stdout
    info = json.loads(out)
    stream = info["streams"][0]
    num, den = stream["r_frame_rate"].split("/")
    return {
        "duration_s": float(info["format"]["duration"]),
        "codec": stream["codec_name"],
        "width": stream["width"],
        "height": stream["height"],
        "fps": round(int(num) / int(den), 2),
    }


rows = []
for v in videos:
    m = re.match(r"(A0\d)_(\d{14})", v.name)
    info = probe(v)
    rows.append({
        "file": v.relative_to(RAW).as_posix(),
        "camera": m.group(1) if m else "",
        # from the file name only: confirm against the CCTV overlay (clock_start_override)
        "name_start": datetime.strptime(m.group(2), "%Y%m%d%H%M%S").isoformat(sep=" ") if m else "",
        "clock_start_override": "",
        **info,
        "hours": round(info["duration_s"] / 3600, 2),
        "expected_sampled_frames": math.ceil(info["duration_s"] / SAMPLE_INTERVAL_SECONDS),
        "chunks": math.ceil(info["duration_s"] / CHUNK_SECONDS),
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFESTS / "video_manifest.csv", index=False)
manifest

In [ ]:
from IPython.display import Image, display


def show_frame(rel_path, elapsed_s):
    """Show one frame; rel_path is relative to data/raw, e.g. 'cam4/A04_20260405000000.mp4'."""
    out = "/content/preview.jpg"
    subprocess.run(
        ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
         "-ss", str(elapsed_s), "-i", str(RAW / rel_path),
         "-frames:v", "1", "-q:v", "3", out],
        check=True,
    )
    display(Image(out))


# Example: the A04 water-bowl event at 00:40:35 (elapsed 2435 s)
# show_frame("cam4/A04_20260405000000.mp4", 2435)